In [ ]:
import pandas as pd
import numpy as np
from pyexplainer import pyexplainer_pyexplainer
from IPython.display import display, HTML


In [ ]:
df = pyexplainer_pyexplainer.load_sample_data()
df.head()

In [ ]:
# Prepare the data
df = df.set_index(df["File"])
df = df.drop(["File", "HeuBug", "HeuBugCount", "RealBugCount"], axis=1)
df.head(3)

In [ ]:
df['RealBug'].value_counts()

In [ ]:
from pyexplainer.pyexplainer_pyexplainer import AutoSpearman

# Select all rows and all feature columns
# the last col which is label col is not selected
X = df.iloc[:, :-1]  # removing the class or result or dependent var column
total_features = len(X.columns)
print(f"Total feature before autospearman: {total_features}")

# apply feature selection function to our feature DataFrame
X = AutoSpearman(X)
selected = len(X.columns)
print(f"After autospearman: {selected}")

# select all rows and the last label column which is the dependent var column
y = df.iloc[:, -1]


# Display the DataFrame with a label
display(HTML("<h3>Selected Features DataFrame</h3>"))
display(X.head(3))
display(HTML("<h3>Target Column</h3>"))
display(y.head(3))

In [ ]:
# Split data into train and test set
from sklearn.model_selection import train_test_split
# 70% training and 30% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

## Training and Predicting

### Train a RandomForest model using sklearn

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_split=2, random_state=0)
rf_model.fit(X_train, y_train)

### Generate Prediction

In [ ]:
# generate prediction from the model which will return a list of predicted labels
y_preds = rf_model.predict(X_test)
# create a DataFrame which only has predicted label column
y_preds = pd.DataFrame(data={'PredictedBug': y_preds}, index=y_test.index)
y_preds.head(3)

## Prediction Post Processing
With only prediction we cannot generate the explanation. Which is why we need to combine the features with the predicted values. Then we will filter out instance where the model prediction was wrong. Because we need to understand from the explanation which features contributed the most during the prediction. 
 

### Combine feature cols, label col and the predicted col in testing set

In [ ]:
print(type(y_test))
combined_testing_data = X_test.join(y_test.to_frame())
combined_testing_data = combined_testing_data.join(y_preds)
combined_testing_data.head(3)
total_rows = len(combined_testing_data)
total_rows

### Filter out wrongly predicted rows

In [ ]:
correctly_predicted_data = combined_testing_data[
    combined_testing_data["RealBug"] == combined_testing_data["PredictedBug"]
]

correctly_predicted_rows = len(correctly_predicted_data)
print(
    f"The model correctly predicted {round(correctly_predicted_rows/total_rows, 3)*100} % of testing data"
)

We focus on the bug file so filter out the non buggy files

In [ ]:
correctly_predicted_bug = correctly_predicted_data[
    correctly_predicted_data["RealBug"] == True
]
len(correctly_predicted_bug)

### Define label and feature columns using the new prepared correctly predicted bug dataset

In [ ]:
# Just select all columns except the last two RealBug and PredictedBug
feature_cols = correctly_predicted_data.iloc[:, :-2]
# As the label select one of the remaining two column as both of them are same
label_col = correctly_predicted_data.iloc[:, -2]


### Select one row of correctly predicted bug to be explained

In [ ]:
# Decide which row to be selected
selected_row = 0
#  select the row in X_test which contains all of the feature values
X_explain = feature_cols.iloc[[selected_row]]
display(X_explain)
# Select the corresponding label from the DataFrame that we just created above
y_explain = label_col.iloc[[selected_row]]
y_explain 

## Create rules using pyexplainer and visualize it

### Initialize PyExplainer object

In [ ]:
from pyexplainer import pyexplainer_pyexplainer

py_explainer = pyexplainer_pyexplainer.PyExplainer(
    X_train=X_train,
    y_train=y_train,
    indep=X_train.columns,
    dep="RealBug",
    blackbox_model=rf_model,
)

### Create rules by triggering explain function under PyExplainer object

This operation is time consuiming as it works with local model?


In [ ]:
rules = py_explainer.explain(
    X_explain=X_explain, y_explain=y_explain, search_function="CrossoverInterpolation"
)

In [ ]:
rules.keys()

In [ ]:
py_explainer.visualise(rules)